# Carga ETL de un almacén de eventos sísmicos

Base de Datos II · Ingeniería Geomática

Ejecute el DDL antes de conectarse. Abra el notebook con la carpeta de estudiantes como directorio de trabajo.

In [ ]:
from collections import Counter
from contextlib import closing
from datetime import datetime, timezone
from decimal import Decimal
from getpass import getpass
from pathlib import Path

from mysql.connector import connect
from pygrametl import ConnectionWrapper
from pygrametl.datasources import CSVSource
from pygrametl.tables import CachedDimension, FactTable

LOTE = "control"  # Cambiar a "practica" después de vaciar el almacén.
RUTA = Path("datos") / LOTE
CONEXION = dict(host="127.0.0.1", port=3306, user="root", database="dw_sismicidad_etl")
ESTADOS = ("asignada", "sin_coincidencia", "multiple")
SIN_COMUNA = {
    "codigo_comuna": "SIN_COMUNA",
    "comuna": "Sin comuna asignada",
    "provincia": None,
    "region": None,
}

## 1. Extracción

Inspeccione los encabezados. Explique cómo se enlazan las tres fuentes.

In [ ]:
def leer_csv(ruta):
    with ruta.open(encoding="utf-8", newline="") as archivo:
        return list(CSVSource(f=archivo, delimiter=","))

eventos = leer_csv(RUTA / "eventos_usgs.csv")
comunas = leer_csv(RUTA / "comunas_bcn.csv")
asociaciones = leer_csv(RUTA / "evento_comuna.csv")

for fuente in (eventos, comunas, asociaciones):
    print(len(fuente), fuente[0])

## 2. Transformación

Construya un índice de asociaciones por evento. Distinga repeticiones exactas y conflictos de identificador. Derive los atributos de fecha en UTC y convierta las medidas. Mantenga el estado de la asociación y asigne el código reservado cuando corresponda.

In [ ]:
def transformar(eventos, comunas, asociaciones):
    """Devuelve las filas transformadas y el número de repeticiones exactas."""
    # Completar.
    raise NotImplementedError

In [ ]:
filas, duplicados = transformar(eventos, comunas, asociaciones)
esperados = Counter(fila["estado_asociacion"] for fila in filas)
assert len(eventos) == len(filas) + duplicados
print(f"Leídas: {len(eventos)} | duplicadas: {duplicados} | únicas: {len(filas)}")
print(dict(esperados))

## 3. Carga

Use las definiciones de tablas siguientes. Precargue territorios, busque o cree el día con ensure y obtenga la clave de territorio con lookup. En esta implementación, keyrefs identifica el hecho por id_evento. El argumento measures de la API enumera las demás columnas que se insertan. Las claves foráneas y los atributos descriptivos del evento conservan su papel en el modelo.

In [ ]:
def preparar_tablas(destino):
    fecha = CachedDimension(
        name="dim_fecha",
        key="fecha_key",
        attributes=["fecha", "dia", "mes", "trimestre", "anio"],
        lookupatts=["fecha"],
        targetconnection=destino,
    )
    territorio = CachedDimension(
        name="dim_territorio",
        key="territorio_key",
        attributes=["codigo_comuna", "comuna", "provincia", "region"],
        lookupatts=["codigo_comuna"],
        targetconnection=destino,
    )
    eventos = FactTable(
        name="fact_evento",
        keyrefs=["id_evento"],
        measures=[
            "fecha_key", "territorio_key", "fecha_hora_utc", "latitud",
            "longitud", "profundidad_km", "magnitud", "tipo_magnitud",
            "estado_asociacion",
        ],
        targetconnection=destino,
    )
    return fecha, territorio, eventos

In [ ]:
def cargar(conexion, comunas, filas, esperados):
    """Carga en una transacción y devuelve el conteo por estado."""
    # Compruebe que el destino esté vacío.
    # Cree el ConnectionWrapper y obtenga las tres tablas.
    # Cargue territorios y hechos.
    # Compare el GROUP BY del destino con esperados antes de confirmar.
    # Revierta la transacción ante un error.
    raise NotImplementedError

In [ ]:
with closing(connect(**CONEXION, password=getpass("Contraseña MySQL: "),
                     autocommit=False, buffered=True)) as conexion:
    print(cargar(conexion, comunas, filas, esperados))

## 4. Validación y consultas

Registre aquí el balance del lote de control y la evidencia de los dos eventos de la misma fecha y comuna. Guarde esa evidencia antes de vaciar el almacén. Luego ejecute el lote de práctica y agregue las consultas SQL y sus resultados.

**Lote de control**

**Lote de práctica**

**Consultas y resultados**

**Decisiones de diseño**